In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [5]:
#Load fine-tuned model
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "FatalHub/qwen-function-calling-10k"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/1.76k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.03k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [6]:
import requests

# Alpha vantage api key
API_KEY = "QGX2CATTTSUI4J54"

In [7]:
def get_stock_price(symbol: str):
    url = "https://www.alphavantage.co/query"

    params = {
        "function": "GLOBAL_QUOTE",
        "symbol": symbol,
        "apikey": API_KEY
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()
    quote = data.get("Global Quote", {})

    if not quote:
        return {
            "error": f"No data found for symbol {symbol}"
        }

    return {
        "symbol": quote.get("01. symbol"),
        "price": float(quote.get("05. price", 0)),
        "change": quote.get("09. change"),
        "change_percent": quote.get("10. change percent")
    }

In [8]:
result = get_stock_price("AAPL")
print(result)

{'symbol': 'AAPL', 'price': 333.69, 'change': '3.3700', 'change_percent': '1.0202%'}


In [9]:
def get_historical_prices(symbol: str, outputsize="compact"):
    url = "https://www.alphavantage.co/query"

    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "outputsize": outputsize,
        "apikey": API_KEY
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()

    time_series = data.get("Time Series (Daily)", {})

    if not time_series:
        return {
            "error": f"No historical data found for {symbol}",
            "raw_response": data
        }

    return {
        "symbol": symbol,
        "data": time_series
    }

In [11]:
result = get_historical_prices("AAPL")

print("Symbol:", result["symbol"])
print("Number of days:", len(result["data"]))

for date, values in list(result["data"].items())[:5]:
    print(date, values)

Symbol: AAPL
Number of days: 100
2026-10-02 {'1. open': '333.2600', '2. high': '334.5400', '3. low': '330.6100', '4. close': '333.6900', '5. volume': '33278552'}
2026-10-01 {'1. open': '330.0000', '2. high': '332.4816', '3. low': '325.8100', '4. close': '330.3200', '5. volume': '36306346'}
2026-09-30 {'1. open': '330.8000', '2. high': '339.5000', '3. low': '330.1401', '4. close': '333.0200', '5. volume': '49988558'}
2026-09-29 {'1. open': '336.9650', '2. high': '337.0850', '3. low': '328.7000', '4. close': '329.4000', '5. volume': '38478037'}
2026-09-28 {'1. open': '340.3700', '2. high': '342.9880', '3. low': '338.0400', '4. close': '338.4000', '5. volume': '32820848'}


In [12]:
def get_company_info(symbol: str):
    url = "https://www.alphavantage.co/query"

    params = {
        "function": "OVERVIEW",
        "symbol": symbol,
        "apikey": API_KEY
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()

    if not data or "Symbol" not in data:
        return {
            "error": f"No company information found for {symbol}",
            "raw_response": data
        }

    return {
        "symbol": data.get("Symbol"),
        "name": data.get("Name"),
        "description": data.get("Description"),
        "sector": data.get("Sector"),
        "industry": data.get("Industry"),
        "market_cap": data.get("MarketCapitalization"),
        "pe_ratio": data.get("PERatio"),
        "eps": data.get("EPS")
    }

result = get_company_info("AAPL")

print("Symbol:", result["symbol"])
print("Name:", result["name"])
print("Sector:", result["sector"])
print("Industry:", result["industry"])
print("Market Cap:", result["market_cap"])
print("P/E:", result["pe_ratio"])
print("EPS:", result["eps"])

Symbol: AAPL
Name: Apple Inc.
Sector: TECHNOLOGY
Industry: CONSUMER ELECTRONICS
Market Cap: 4869931926000
P/E: 38.31
EPS: 8.71


In [13]:
def get_company_news(symbol):
    params = {
        "function": "NEWS_SENTIMENT",
        "tickers": symbol,
        "apikey": ALPHA_VANTAGE_API_KEY
    }

    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()

    data = response.json()

    articles = []

    for article in data.get("feed", [])[:5]:
        articles.append({
            "title": article.get("title"),
            "summary": article.get("summary"),
            "url": article.get("url"),
            "source": article.get("source"),
            "sentiment": article.get("overall_sentiment_label"),
            "sentiment_score": article.get("overall_sentiment_score")
        })

    return {
        "symbol": symbol,
        "articles": articles
    }

In [14]:
def get_exchange_rate(from_currency, to_currency):
    params = {
        "function": "CURRENCY_EXCHANGE_RATE",
        "from_currency": from_currency,
        "to_currency": to_currency,
        "apikey": ALPHA_VANTAGE_API_KEY
    }

    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()

    data = response.json()

    exchange_data = data.get("Realtime Currency Exchange Rate", {})

    return {
        "from_currency": exchange_data.get("1. From_Currency Code"),
        "to_currency": exchange_data.get("3. To_Currency Code"),
        "exchange_rate": exchange_data.get("5. Exchange Rate"),
        "last_updated": exchange_data.get("6. Last Refreshed")
    }

In [15]:
def calculate_percentage_change(old_price: float, new_price: float):
    return ((new_price - old_price) / old_price) * 100


history = get_historical_prices("AAPL")

data = history["data"]

dates = list(data.keys())

latest_date = dates[0]
old_date = dates[29]

latest_close = float(data[latest_date]["4. close"])
old_close = float(data[old_date]["4. close"])

change = calculate_percentage_change(old_close, latest_close)

print(f"Latest date: {latest_date}")
print(f"Latest close: ${latest_close}")
print(f"Date 30 trading days ago: {old_date}")
print(f"Old close: ${old_close}")
print(f"Change: {change:.2f}%")

Latest date: 2026-10-02
Latest close: $333.69
Date 30 trading days ago: 2026-08-21
Old close: $309.35
Change: 7.87%


In [36]:
tools = [
    {
        "name": "get_stock_price",
        "description": "Get the current stock price and daily change for a company.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "The stock ticker symbol, e.g. AAPL, MSFT, TSLA."
                }
            },
            "required": ["symbol"]
        }
    },
    {
        "name": "get_historical_prices",
        "description": "Get daily historical stock prices for a company.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "The stock ticker symbol, e.g. AAPL, MSFT, TSLA."
                }
            },
            "required": ["symbol"]
        }
    },
    {
        "name": "get_company_info",
        "description": "Get general company information and financial metrics.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "The stock ticker symbol, e.g. AAPL, MSFT, TSLA."
                }
            },
            "required": ["symbol"]
        }
    },
    {
        "name": "get_company_news",
        "description": "Get recent news articles and sentiment information for a company.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "The stock ticker symbol, e.g. AAPL, MSFT, TSLA."
                }
            },
            "required": ["symbol"]
        }
    },
    {
        "name": "get_exchange_rate",
        "description": "Get the current exchange rate between two currencies.",
        "parameters": {
            "type": "object",
            "properties": {
                "from_currency": {
                    "type": "string",
                    "description": "The three-letter currency code to convert from, e.g. USD, EUR, GBP."
                },
                "to_currency": {
                    "type": "string",
                    "description": "The three-letter currency code to convert to, e.g. USD, EUR, GBP."
                }
            },
            "required": ["from_currency", "to_currency"]
        }
    }
]

In [43]:
import json
import torch

query = "what is the exchange rate between Egyptian pound and USD?"

prompt = f"""Tools:
{json.dumps(tools, indent=2)}

Query:
{query}

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{"name": "get_exchange_rate", "arguments": {"from_currency": "EGP", "to_currency": "USD"}}]


In [24]:
query = "what is the capital of spain?"

prompt = f"""Tools:
{json.dumps(tools, indent=2)}

Query:
{query}

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


There is no direct API endpoint to query the capital of Spain. However, you can use the 'get_company_info' function to get information about a Spanish company, which might indirectly help in finding the capital. For example:

```python
response = get_company_info(symbol='IBERDORRA')
print(response)
```

This will return information about IBERDORRA, a Spanish company, which could be used to infer that the capital of Spain is Madrid. Note that this is an indirect approach and not a direct answer to the question.


In [41]:
query = "what is the amazon price, tell me about microsoft, what do people say about Nvidia?"

prompt = f"""You are a tool-calling assistant.

Tools:
{json.dumps(tools, indent=2)}

Rules:
1. You may ONLY use tools listed above.
2. If none of the available tools can answer the query, do NOT call any tool.
3. Do NOT invent tool names.
4. Do NOT use a tool for an indirect or unrelated purpose.
5. If no suitable tool exists, answer naturally that no suitable tool is available.
6. Do not explain how to use a tool.
7. For a valid tool call, return only the JSON tool call.

Query:
{query}

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{"name": "get_stock_price", "arguments": {"symbol": "AMZN"}}, {"name": "get_company_info", "arguments": {"symbol": "MSFT"}}, {"name": "get_company_news", "arguments": {"symbol": "NVDA"}}]


In [32]:
query = "what is the whether in Egypt?"

prompt = f"""You are a tool-calling assistant.

Tools:
{json.dumps(tools, indent=2)}

Rules:
1. You may ONLY use tools listed above.
2. If none of the available tools can answer the query, do NOT call any tool.
3. Do NOT invent tool names.
4. Do NOT use a tool for an indirect or unrelated purpose.
5. If no suitable tool exists, answer naturally that no suitable tool is available.
6. Do not explain how to use a tool.
7. For a valid tool call, return only the JSON tool call.

Query:
{query}

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


No suitable tool found.


In [44]:
query = "what is the capital of france?"

prompt = f"""You are a tool-calling assistant.

Tools:
{json.dumps(tools, indent=2)}

Rules:
1. You may ONLY use tools listed above.
2. If none of the available tools can answer the query, do NOT call any tool.
3. Do NOT invent tool names.
4. Do NOT use a tool for an indirect or unrelated purpose.
5. If no suitable tool exists, answer naturally that no suitable tool is available.
6. Do not explain how to use a tool.
7. For a valid tool call, return only the JSON tool call.

Query:
{query}

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


No suitable tool found.


In [37]:
query = "Retrieve information for product '75319' in Australian Dollars and Italian language."

prompt = f"""You are a tool-calling assistant.

Tools:
{json.dumps(tools, indent=2)}

Rules:
1. You may ONLY use tools listed above.
2. If none of the available tools can answer the query, do NOT call any tool.
3. Do NOT invent tool names.
4. Do NOT use a tool for an indirect or unrelated purpose.
5. If no suitable tool exists, answer naturally that no suitable tool is available.
6. Do not explain how to use a tool.
7. For a valid tool call, return only the JSON tool call.

Query:
{query}

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{"name": "get_product_details", "arguments": {"product_id": "75319", "currency": "AUD", "language": "it"}}]


This needs a validation layer